# Chapter 02 - The Object Model, Properly

#### 1. Mechanical Refresher
A Python instance stores its own attributes, and a method call like `obj.method(x)` is mechanically a function call where Python passes `obj` as the first argument named `self`. Class attributes live on the class and are found by instances when the instance does not have that attribute; `classmethod`, `staticmethod`, and `property` are built-in wrappers that change how attribute lookup turns class functions into callable or computed attributes.

#### 2. Minimal Working Example

In [ ]:
class Counter:
    step = 1

    def __init__(self, start):
        self.value = start

    def advance(self):
        self.value = self.value + self.step
        return self.value

counter = Counter(10)
print(Counter.advance(counter))
print(counter.advance())
print(counter.value)

`Counter.advance(counter)` shows the explicit form of what `counter.advance()` does for `self`. `step` is found on the class unless an instance creates its own `step`. `value` is stored directly on each instance.

#### 3. Modify Drills

**Modify Drill 1.** Change the starting value and predict both calls to `advance`.

In [ ]:
class Counter:
    step = 2

    def __init__(self, start):
        self.value = start

    def advance(self):
        self.value = self.value + self.step
        return self.value

counter = Counter(5)
actual = [Counter.advance(counter), counter.advance()]
expected = [7, 9]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 2.** Give only `right` an instance-level `label`; predict which labels each instance sees.

In [ ]:
class Item:
    label = "class-label"

    def __init__(self, name):
        self.name = name

left = Item("left")
right = Item("right")
right.label = "instance-label"
actual = [left.label, right.label, Item.label]
expected = ["class-label", "instance-label", "class-label"]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 3.** Change `prefix` and predict `build` without using decorator syntax yet.

In [ ]:
class BatchName:
    prefix = "batch"

    def build(cls, number):
        return cls.prefix + "-" + str(number)
    build = classmethod(build)

    def is_empty(size):
        return size == 0
    is_empty = staticmethod(is_empty)

actual = [BatchName.build(4), BatchName.is_empty(0)]
expected = ["batch-4", True]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by removing `self` from `rename`. Predict the argument-count error, then restore `self`.

In [ ]:
class Record:
    def __init__(self, name):
        self.name = name

    def rename(self, new_name):
        self.name = new_name
        return self.name

record = Record("old")
actual = record.rename("new")
expected = "new"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Break-and-Fix Drill 2.** Break it by assigning `Bucket.items = []` inside `add`. Predict why separate instances share state, then fix it by using `self.items`.

In [ ]:
class Bucket:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

first = Bucket()
second = Bucket()
first.add("a")
actual = [first.items, second.items]
expected = [["a"], []]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 5. Self-Verification
This chapter still uses expected-vs-actual prints because `assert` has not been built yet. Check both the value and the `match:` line after each edit.

#### 6. Standalone Exercises

**Exercise 1.** Fill in `full_name` using `self.first` and `self.last`. Expected behavior: `'Ada Lovelace'`.

In [ ]:
class Person:
    def __init__(self, first, last):
        self.first = first
        self.last = last

    def full_name(self):
        return None  # TODO

actual = Person("Ada", "Lovelace").full_name()
expected = "Ada Lovelace"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 2.** Add only to the instance attribute. Expected behavior: `[['x'], []]`.

In [ ]:
class Log:
    default_name = "run"

    def __init__(self):
        self.entries = []

    def add(self, entry):
        # TODO: append to the instance's entries.
        return None

a = Log()
b = Log()
a.add("x")
actual = [a.entries, b.entries]
expected = [["x"], []]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 3.** Create a property manually. Expected behavior: reading `area` returns `12`.

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    # TODO: replace area with property(area).

actual = Rectangle(3, 4).area
expected = 12
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 4.** Create a `classmethod` named `from_pair`. Expected behavior: the object stores `left=2` and `right=5`.

In [ ]:
class Pair:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def from_pair(cls, pair):
        return cls(pair[0], pair[1])

    # TODO: wrap from_pair with classmethod.

actual_obj = Pair(0, 0)  # TODO: after wrapping, replace with Pair.from_pair((2, 5)).
actual = [actual_obj.left, actual_obj.right]
expected = [2, 5]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 5.** Create a `staticmethod` named `valid_size`. Expected behavior: `[False, True]`.

In [ ]:
class Batch:
    def valid_size(size):
        return size > 0

    # TODO: wrap valid_size with staticmethod.

actual = [False, False]  # TODO: after wrapping, call Batch().valid_size for 0 and 3.
expected = [False, True]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 7. Applied AI/ML Drill
**ML to Python mirror:** a small model-configuration object is just an instance with attributes and methods that compute from those attributes. **Python to ML mirror:** framework objects such as layers, optimizers, and configs rely on the same `self` mechanics; a layer method reading `self.weight` is the same instance-attribute lookup practiced here.

**Applied Drill.** Complete the computed `parameter_count` property without decorator syntax. Expected behavior: a linear layer with 3 inputs and 2 outputs has 8 parameters.

In [ ]:
class LinearConfig:
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def parameter_count(self):
        # TODO: weights plus bias values.
        return None

    parameter_count = property(parameter_count)

config = LinearConfig(3, 2)
actual = config.parameter_count
expected = 8
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 8. Common Bugs
- Missing `self`: instance method calls pass the instance automatically; the symptom is an argument-count `TypeError`.
- Writing to the class when you meant the instance: the symptom is state leaking across objects.
- Confusing `obj.method` with `obj.method()`: the first is a bound method object; the second calls it and returns its result.
- Forgetting that property access has no parentheses: the symptom is either a method object printed or a `'int' object is not callable` error after adding extra parentheses.

#### 9. Compounding Drill

Combine Chapter 1 function objects with this chapter's object model: store a metric function on an evaluator instance and call it from a method.

In [ ]:
def absolute_error(pred, target):
    return abs(pred - target)

class Evaluator:
    def __init__(self, metric_fn):
        self.metric_fn = metric_fn

    def evaluate_one(self, pred, target):
        # TODO: call the stored metric function.
        return None

evaluator = Evaluator(absolute_error)
actual = evaluator.evaluate_one(10, 6)
expected = 4
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 10. Chapter Project
No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Modify drills
class Counter:
    step = 2

    def __init__(self, start):
        self.value = start

    def advance(self):
        self.value = self.value + self.step
        return self.value

counter = Counter(10)
print([Counter.advance(counter), counter.advance()])

class Item:
    label = "class-label"

    def __init__(self, name):
        self.name = name

left = Item("left")
right = Item("right")
right.label = "instance-label"
print([left.label, right.label, Item.label])

In [ ]:
# classmethod, staticmethod, and property solutions
class BatchName:
    prefix = "run"

    def build(cls, number):
        return cls.prefix + "-" + str(number)
    build = classmethod(build)

    def is_empty(size):
        return size == 0
    is_empty = staticmethod(is_empty)

print(BatchName.build(4), BatchName.is_empty(0))

class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height
    area = property(area)

print(Rectangle(3, 4).area)

In [ ]:
# Standalone Exercises 1-2
class Person:
    def __init__(self, first, last):
        self.first = first
        self.last = last

    def full_name(self):
        return self.first + " " + self.last

print(Person("Ada", "Lovelace").full_name())

class Log:
    default_name = "run"

    def __init__(self):
        self.entries = []

    def add(self, entry):
        self.entries.append(entry)

a = Log()
b = Log()
a.add("x")
print([a.entries, b.entries])

In [ ]:
# Standalone Exercises 4-5, Applied Drill, and Compounding Drill
class Pair:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def from_pair(cls, pair):
        return cls(pair[0], pair[1])
    from_pair = classmethod(from_pair)

pair = Pair.from_pair((2, 5))
print(pair.left, pair.right)

class Batch:
    def valid_size(size):
        return size > 0
    valid_size = staticmethod(valid_size)

print([Batch.valid_size(0), Batch.valid_size(3)])

class LinearConfig:
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def parameter_count(self):
        return self.inputs * self.outputs + self.outputs
    parameter_count = property(parameter_count)

print(LinearConfig(3, 2).parameter_count)

def absolute_error(pred, target):
    return abs(pred - target)

class Evaluator:
    def __init__(self, metric_fn):
        self.metric_fn = metric_fn

    def evaluate_one(self, pred, target):
        return self.metric_fn(pred, target)

print(Evaluator(absolute_error).evaluate_one(10, 6))